In [0]:
# ============================================================
# STEP 5 — Full Interactive Dashboard with Heatmaps
# ============================================================
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

# ── Load Data ──
pred_df = spark.table("workspace.default.yield_predictions_new").toPandas()
corn = pred_df[pred_df["commodity_name"] == "Corn"].copy()
soy  = pred_df[pred_df["commodity_name"] == "Soybeans"].copy()
corn["uncertainty"] = corn["yield_q90"] - corn["yield_q10"]
soy["uncertainty"]  = soy["yield_q90"]  - soy["yield_q10"]

# Load state/county info
features_df = spark.table("workspace.default.corn_soy_yield_weather_features") \
    .select("county_name", "state_code", "state_abbreviation", "county_code") \
    .distinct().toPandas()

# Merge and build FIPS
def build_fips(df):
    df = df.merge(features_df, on="county_name", how="left")
    df["state_code"]  = df["state_code"].astype(str).str.zfill(2)
    df["county_code"] = df["county_code"].astype(str).str.zfill(3)
    df["fips"]        = df["state_code"] + df["county_code"]
    return df

corn = build_fips(corn)
soy  = build_fips(soy)

def agg_county(df):
    return df.groupby(["fips", "county_name", "state_abbreviation"]).agg(
        q50=("yield_q50", "mean"),
        q10=("yield_q10", "mean"),
        q90=("yield_q90", "mean"),
        actual=("yield_actual", "mean"),
        uncertainty=("uncertainty", "mean")
    ).reset_index().assign(
        uncertainty_pct=lambda x: ((x["q90"] - x["q10"]) / x["q50"] * 100).round(1)
    )

corn_map = agg_county(corn)
soy_map  = agg_county(soy)

GEOJSON = "https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json"

# ============================================================
# MAP 1 — Corn Yield Forecast (Toggle Q10 / Q50 / Q90)
# ============================================================
fig1 = go.Figure()

for q, label, visible in [("q10", "Pessimistic (Q10)", False),
                            ("q50", "Median (Q50)",     True),
                            ("q90", "Optimistic (Q90)", False)]:
    fig1.add_trace(go.Choropleth(
        geojson=GEOJSON,
        locations=corn_map["fips"],
        z=corn_map[q],
        colorscale="RdYlGn",
        zmin=corn_map["q10"].min(),
        zmax=corn_map["q90"].max(),
        visible=visible,
        name=label,
        colorbar_title="bu/acre",
        hovertemplate="<b>%{text}</b><br>" + label + ": %{z:.1f} bu/acre<extra></extra>",
        text=corn_map["county_name"] + ", " + corn_map["state_abbreviation"]
    ))

fig1.update_layout(
    title="🌽 Corn Yield Forecast — Toggle Between Scenarios",
    geo_scope="usa",
    height=600,
    updatemenus=[dict(
        type="buttons",
        direction="right",
        x=0.5, y=1.08,
        xanchor="center",
        buttons=[
            dict(label="😟 Pessimistic (Q10)", method="update",
                 args=[{"visible": [True, False, False]}]),
            dict(label="📊 Median (Q50)",      method="update",
                 args=[{"visible": [False, True, False]}]),
            dict(label="😊 Optimistic (Q90)",  method="update",
                 args=[{"visible": [False, False, True]}]),
        ]
    )]
)
fig1.show()

# ============================================================
# MAP 2 — Risk Heatmap (Uncertainty %) with Slider by Year
# ============================================================
corn_year = build_fips(pred_df[pred_df["commodity_name"] == "Corn"].copy())
corn_year["uncertainty_pct"] = ((corn_year["yield_q90"] - corn_year["yield_q10"]) / corn_year["yield_q50"] * 100).round(1)

years = sorted(corn_year["yield_year"].unique())
frames = []
for yr in years:
    df_yr = corn_year[corn_year["yield_year"] == yr].groupby(["fips", "county_name", "state_abbreviation"]).agg(
        uncertainty_pct=("uncertainty_pct", "mean")
    ).reset_index()
    frames.append(go.Frame(
        data=[go.Choropleth(
            geojson=GEOJSON,
            locations=df_yr["fips"],
            z=df_yr["uncertainty_pct"],
            colorscale="Reds",
            zmin=0, zmax=80,
            text=df_yr["county_name"] + ", " + df_yr["state_abbreviation"],
            hovertemplate="<b>%{text}</b><br>Uncertainty: %{z:.1f}%<extra></extra>"
        )],
        name=str(yr)
    ))

df_first = corn_year[corn_year["yield_year"] == years[0]].groupby(["fips", "county_name", "state_abbreviation"]).agg(
    uncertainty_pct=("uncertainty_pct", "mean")
).reset_index()

fig2 = go.Figure(
    data=[go.Choropleth(
        geojson=GEOJSON,
        locations=df_first["fips"],
        z=df_first["uncertainty_pct"],
        colorscale="Reds",
        zmin=0, zmax=80,
        colorbar_title="Uncertainty %",
        text=df_first["county_name"] + ", " + df_first["state_abbreviation"],
        hovertemplate="<b>%{text}</b><br>Uncertainty: %{z:.1f}%<extra></extra>"
    )],
    frames=frames
)
fig2.update_layout(
    title="⚠️ Corn Yield Risk Heatmap by Year — Drag Slider to Animate",
    geo_scope="usa",
    height=600,
    sliders=[dict(
        steps=[dict(method="animate", args=[[str(yr)],
               dict(mode="immediate", frame=dict(duration=500, redraw=True))],
               label=str(yr)) for yr in years],
        x=0.1, y=0, len=0.9,
        currentvalue=dict(prefix="Year: ", font=dict(size=14))
    )],
    updatemenus=[dict(
        type="buttons", x=0.05, y=-0.08,
        buttons=[
            dict(label="▶ Play",  method="animate",
                 args=[None, dict(frame=dict(duration=600, redraw=True), fromcurrent=True)]),
            dict(label="⏸ Pause", method="animate",
                 args=[[None], dict(mode="immediate", frame=dict(duration=0))])
        ]
    )]
)
fig2.show()

# ============================================================
# CHART 3 — Heatmap: Avg Yield by State × Year
# ============================================================
state_year = corn.groupby(["state_abbreviation", "yield_year"]).agg(
    q50=("yield_q50", "mean")
).reset_index().pivot(index="state_abbreviation", columns="yield_year", values="q50")

fig3 = px.imshow(
    state_year,
    color_continuous_scale="RdYlGn",
    title="🌡️ Corn Yield Heatmap — State × Year",
    labels=dict(x="Year", y="State", color="Yield (bu/acre)"),
    aspect="auto"
)
fig3.update_layout(height=700, template="plotly_white")
fig3.update_traces(hovertemplate="<b>%{y}</b><br>Year: %{x}<br>Yield: %{z:.1f} bu/acre<extra></extra>")
fig3.show()

# ============================================================
# CHART 4 — Scenario Impact Heatmap by State
# ============================================================
baseline  = corn.groupby("state_abbreviation")["yield_q50"].mean()
drought   = baseline * 0.88
heatwave  = baseline * 0.85
exrain    = baseline * 0.93

scenario_heat = pd.DataFrame({
    "Baseline":             baseline,
    "Drought (-30% precip)":      drought,
    "Heatwave (+5°C)":            heatwave,
    "Excess Rain (+50% precip)":  exrain
}).T

fig4 = px.imshow(
    scenario_heat,
    color_continuous_scale="RdYlGn",
    title="🌦️ Scenario Impact Heatmap — Yield by State & Scenario",
    labels=dict(x="State", y="Scenario", color="Yield (bu/acre)"),
    aspect="auto"
)
fig4.update_layout(height=400, template="plotly_white")
fig4.update_traces(hovertemplate="<b>%{y}</b><br>State: %{x}<br>Yield: %{z:.1f} bu/acre<extra></extra>")
fig4.show()

# ============================================================
# CHART 5 — Uncertainty Band Chart (Top 20 Counties)
# ============================================================
top20 = corn_map.nlargest(20, "uncertainty_pct").sort_values("uncertainty_pct")

fig5 = go.Figure()
fig5.add_trace(go.Bar(
    y=top20["county_name"] + ", " + top20["state_abbreviation"],
    x=top20["q90"] - top20["q10"],
    base=top20["q10"],
    orientation="h",
    name="Uncertainty Band (Q10–Q90)",
    marker_color="rgba(243, 156, 18, 0.6)",
    hovertemplate="<b>%{y}</b><br>Q10: %{base:.1f}<br>Q90: %{x:.1f} bu/acre<extra></extra>"
))
fig5.add_trace(go.Scatter(
    y=top20["county_name"] + ", " + top20["state_abbreviation"],
    x=top20["q50"],
    mode="markers",
    name="Median (Q50)",
    marker=dict(color="#e74c3c", size=12, symbol="diamond"),
    hovertemplate="<b>%{y}</b><br>Median: %{x:.1f} bu/acre<extra></extra>"
))
fig5.update_layout(
    title="⚠️ Top 20 Highest Risk Counties — Yield Uncertainty Range",
    xaxis_title="Yield (bu/acre)",
    height=650,
    template="plotly_white",
    legend=dict(orientation="h", y=1.05)
)
fig5.show()

print("✅ All 5 interactive charts complete!")
print("   Map 1   — Toggle Q10/Q50/Q90 forecast map")
print("   Map 2   — Animated risk heatmap with year slider")
print("   Chart 3 — State × Year yield heatmap")
print("   Chart 4 — Scenario impact heatmap by state")
print("   Chart 5 — Top 20 riskiest counties uncertainty bands")

✅ All 5 interactive charts complete!
   Map 1   — Toggle Q10/Q50/Q90 forecast map
   Map 2   — Animated risk heatmap with year slider
   Chart 3 — State × Year yield heatmap
   Chart 4 — Scenario impact heatmap by state
   Chart 5 — Top 20 riskiest counties uncertainty bands
